## 2단계 Multi Chain 만들기 - 여행지 정보 시스템

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 기본 LLM 제공자: "groq" 또는 "openai" — get_llm() 호출 시 provider 를 생략하면 이 값을 사용
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

# 1단계: 여행지 → 대표 명소 1개 추천
## 직업 -> 해야할 일 -> 답변 형태 -> 예시
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 세계 여행지를 잘 아는 여행 전문가입니다. "
     "사용자가 도시나 국가를 입력하면, 그 지역의 가장 대표적인 관광 명소 딱 1곳을 추천합니다. "
     "설명이나 인사말 없이 명소 이름만 한 줄로 답하세요. "
     "예시) 입력: 파리 → 출력: 에펠탑"),
    ("user", "여행지: {destination}"),
])

# 2단계: 명소 → 상세 정보 제공
detail_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 친절한 여행 가이드입니다. "
     "사용자가 관광 명소 이름을 알려주면, 여행자에게 도움이 되도록 아래 항목을 정리해 설명합니다.\n"
     "1. 명소 소개 (어떤 곳인지)\n"
     "2. 역사와 의미\n"
     "3. 볼거리와 즐길 거리\n"
     "4. 방문 팁 (추천 시간대, 주의사항 등)\n"
     "정확하지 않은 정보(입장료, 운영시간 등)는 지어내지 말고, 확인이 필요하다고 안내하세요."),
    ("user", "관광 명소: {attraction}"),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
llm = get_llm(temperature=0.7, provider="groq") 
response = llm.invoke(messages)

print(type(response))
print(response.content)